# 01 · Bronze – Ingestão SIDRA/IBGE (Fabric)

Cópia fiel da fonte: nada é tipado, renomeado ou filtrado. A linha de cabeçalho do SIDRA (`_linha = 0`) é mantida e removida só na Silver.

- **Paginação por ano**: uma chamada por período, para não estourar o limite de valores da API.
- **Idempotente**: cada execução reescreve as tabelas (overwrite). Rodar de novo não duplica.
- **Auditável**: JSON original em `Files/bronze/raw/`, colunas de auditoria e log em `Files/logs/`.

**Antes de rodar:** anexe o Lakehouse `lh_panorama` como lakehouse padrão (painel *Explorer* à esquerda).

In [1]:
# Parâmetros (mesmos valores de pipeline/config.py da versão local)
FONTES = {
    "populacao": {"tabela": 6579, "variavel": 9324, "nivel": "n6"},  # pop. residente estimada
    "pib":       {"tabela": 5938, "variavel": 37,   "nivel": "n6"},  # PIB a preços correntes (mil R$)
}
ANOS = None

SIDRA_BASE = "https://apisidra.ibge.gov.br/values"
AGREGADOS_BASE = "https://servicodados.ibge.gov.br/api/v3/agregados"
TIMEOUT_S, MAX_TENTATIVAS, PAUSA_S = 120, 4, 1.0

# Caminho local do lakehouse padrão montado no notebook
FILES = "/lakehouse/default/Files"


StatementMeta(, f6bf7c45-8178-4c9e-8c1a-a62de38658d9, 3, Finished, Available, Finished, False)

In [2]:
import json, os, time
from datetime import datetime, timezone
import pandas as pd
import requests

LOG = []
def registrar(msg):
    linha = f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} | {msg}"
    print(linha); LOG.append(linha)

def get_json(url):
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            r = requests.get(url, timeout=TIMEOUT_S, headers={"User-Agent": "panorama-municipal/0.1"})
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            registrar(f"Tentativa {tentativa}/{MAX_TENTATIVAS} falhou: {e}")
            time.sleep(2 ** tentativa)
    raise RuntimeError(f"Falha definitiva: {url}")

#def listar_periodos(tabela):
  #    return [p["id"] for p in get_json(f"{AGREGADOS_BASE}/{tabela}/periodos")]

def listar_periodos(tabela, variavel):
    """Anos disponíveis, pela própria API SIDRA (nível Brasil, todos os períodos)."""
    dados = get_json(f"{SIDRA_BASE}/t/{tabela}/n1/all/v/{variavel}/p/all")
    cabecalho, linhas = dados[0], dados[1:]
    col_ano = next(k for k, v in cabecalho.items() if v == "Ano (Código)")
    return sorted({linha[col_ano] for linha in linhas})

def baixar_periodo(nome, cfg, ano):
    url = f"{SIDRA_BASE}/t/{cfg['tabela']}/{cfg['nivel']}/all/v/{cfg['variavel']}/p/{ano}"
    registros = get_json(url)

    pasta = f"{FILES}/bronze/raw/{nome}"
    os.makedirs(pasta, exist_ok=True)
    with open(f"{pasta}/{ano}.json", "w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False)

    df = pd.DataFrame(registros).astype(str)
    df.insert(0, "_linha", range(len(df)))  # 0 = cabeçalho do SIDRA
    df["_fonte"] = nome
    df["_tabela_sidra"] = str(cfg["tabela"])
    df["_periodo_requisitado"] = str(ano)
    df["_url_origem"] = url
    df["_ingerido_em"] = datetime.now(timezone.utc).isoformat()

    registrar(f"{nome} | {ano} | {len(df) - 1} linhas de dados")
    if len(df) > 1:
        registrar(f"{nome} | variável retornada: {df.loc[1].get('D3N', df.loc[1].get('D2N'))}")
    return df


StatementMeta(, f6bf7c45-8178-4c9e-8c1a-a62de38658d9, 4, Finished, Available, Finished, False)

In [3]:
resumo = {}
for nome, cfg in FONTES.items():
    #periodos = ANOS or listar_periodos(cfg["tabela"])
    periodos = ANOS or listar_periodos(cfg["tabela"], cfg["variavel"])
    registrar(f"{nome} | tabela {cfg['tabela']} | períodos: {', '.join(periodos)}")

    partes, falhas = [], []
    for ano in periodos:
        try:
            partes.append(baixar_periodo(nome, cfg, ano))
        except Exception as e:
            registrar(f"{nome} | {ano} | FALHA: {e}")
            falhas.append(ano)
        time.sleep(PAUSA_S)

    if not partes:
        raise RuntimeError(f"Nenhum período ingerido para {nome}")

    pdf = pd.concat(partes, ignore_index=True)
    (spark.createDataFrame(pdf)
          .write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(f"bronze_{nome}"))

    resumo[nome] = {"linhas_total": len(pdf), "periodos": len(periodos), "falhas": falhas}
    registrar(f"{nome} | tabela bronze_{nome} gravada | {len(pdf)} linhas | falhas: {falhas or 'nenhuma'}")

# Log da execução em Files/logs
os.makedirs(f"{FILES}/logs", exist_ok=True)
with open(f"{FILES}/logs/bronze_{datetime.now():%Y%m%d_%H%M%S}.log", "w", encoding="utf-8") as f:
    f.write("\n".join(LOG))
resumo


StatementMeta(, f6bf7c45-8178-4c9e-8c1a-a62de38658d9, 5, Finished, Available, Finished, False)

2026-11-15 19:15:40 | populacao | tabela 6579 | períodos: 2001, 2002, 2003, 2004, 2005, 2006, 2008, 2009, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2024, 2025, 2026
2026-11-15 19:15:41 | populacao | 2001 | 5571 linhas de dados
2026-11-15 19:15:41 | populacao | variável retornada: 2001
2026-11-15 19:15:43 | populacao | 2002 | 5571 linhas de dados
2026-11-15 19:15:43 | populacao | variável retornada: 2002
2026-11-15 19:15:46 | populacao | 2003 | 5571 linhas de dados
2026-11-15 19:15:46 | populacao | variável retornada: 2003
2026-11-15 19:15:48 | populacao | 2004 | 5571 linhas de dados
2026-11-15 19:15:48 | populacao | variável retornada: 2004
2026-11-15 19:15:51 | populacao | 2005 | 5571 linhas de dados
2026-11-15 19:15:51 | populacao | variável retornada: 2005
2026-11-15 19:15:53 | populacao | 2006 | 5571 linhas de dados
2026-11-15 19:15:53 | populacao | variável retornada: 2006
2026-11-15 19:15:55 | populacao | 2008 | 5571 linhas de dados
2026-11-15 19:15:55 | p

{'populacao': {'linhas_total': 122584, 'periodos': 22, 'falhas': []},
 'pib': {'linhas_total': 122562, 'periodos': 22, 'falhas': []}}

In [4]:
# Conferência rápida: linhas de dados por ano em cada tabela (sem o cabeçalho)
for nome in FONTES:
    print(f"== bronze_{nome}")
    spark.sql(f"""
        SELECT _periodo_requisitado AS ano, COUNT(*) AS linhas
        FROM bronze_{nome}
        WHERE _linha > 0
        GROUP BY _periodo_requisitado
        ORDER BY ano
    """).show(50)


StatementMeta(, f6bf7c45-8178-4c9e-8c1a-a62de38658d9, 6, Finished, Available, Finished, False)

== bronze_populacao
+----+------+
| ano|linhas|
+----+------+
|2001|  5571|
|2002|  5571|
|2003|  5571|
|2004|  5571|
|2005|  5571|
|2006|  5571|
|2008|  5571|
|2009|  5571|
|2011|  5571|
|2012|  5571|
|2013|  5571|
|2014|  5571|
|2015|  5571|
|2016|  5571|
|2017|  5571|
|2018|  5571|
|2019|  5571|
|2020|  5571|
|2021|  5571|
|2024|  5571|
|2025|  5571|
|2026|  5571|
+----+------+

== bronze_pib
+----+------+
| ano|linhas|
+----+------+
|2002|  5570|
|2003|  5570|
|2004|  5570|
|2005|  5570|
|2006|  5570|
|2007|  5570|
|2008|  5570|
|2009|  5570|
|2010|  5570|
|2011|  5570|
|2012|  5570|
|2013|  5570|
|2014|  5570|
|2015|  5570|
|2016|  5570|
|2017|  5570|
|2018|  5570|
|2019|  5570|
|2020|  5570|
|2021|  5570|
|2022|  5570|
|2023|  5570|
+----+------+



In [5]:
for nome in ["populacao", "pib"]:
    print(f"== bronze_{nome}")
    spark.sql(f"SELECT * FROM bronze_{nome} WHERE _linha IN (0, 1)").show(truncate=False, vertical=True)

StatementMeta(, f6bf7c45-8178-4c9e-8c1a-a62de38658d9, 7, Finished, Available, Finished, False)

== bronze_populacao
-RECORD 0-------------------------------------------------------------------------------
 _linha               | 0                                                               
 NC                   | Nível Territorial (Código)                                      
 NN                   | Nível Territorial                                               
 MC                   | Unidade de Medida (Código)                                      
 MN                   | Unidade de Medida                                               
 V                    | Valor                                                           
 D1C                  | Município (Código)                                              
 D1N                  | Município                                                       
 D2C                  | Variável (Código)                                               
 D2N                  | Variável                                                        
 